# Week 1: What is GIS? Coordinate systems and datums

**Track:** Ground Station Operator (Beginner)  
**Full primer + quiz:** [https://launchdetect.com/academy/week/1/](https://launchdetect.com/academy/week/1/)  
**Track index:** [https://launchdetect.com/academy/ground-station-operator/](https://launchdetect.com/academy/ground-station-operator/)

---

_GIS is more than maps — it's the science of geographic information. This week covers the foundation: coordinate systems, datums, latitude/longitude, and why getting these right matters in space GIS where coordinates come from satellites._


## Why this week matters

Every space-domain measurement you'll ever touch comes attached to a coordinate, and every coordinate is meaningless without two answers: **which system is it in**, and **what does that system measure**. Get either wrong and your downstream maps, distances, and detections are silently wrong — not crash-and-burn wrong, but off-by-tens-of-kilometers wrong, which is far more dangerous because nothing in the code complains.

Space GIS amplifies this in a brutal way: your coordinates come from satellites, which natively report in geocentric or sensor-local frames; you visualize them on screens that are flat (so projected); and you compare them to ground-station registries that may have been compiled in a regional datum from the 1980s. Three different coordinate systems on three sides of one query. Get fluent now or drown later.

This is why Week 1 is the entire foundation. Every subsequent week assumes you instinctively know which CRS you're in and which one you're going to.


## Learning objectives

By the end of this lab you will be able to:

- Explain what GIS is and isn't
- Distinguish geographic (lat/lon) from projected coordinate systems
- Identify WGS84 as the datum behind GPS and satellite-derived data
- Avoid the most common coordinate-system mistakes in space GIS


## Setup — and why these dependencies

**`leafmap`** — the universal interactive-map widget for Jupyter / Colab. Wraps Folium + ipyleaflet + MapLibre under one API. We use it because a map you can actually see is worth ten map objects you can't, and `leafmap` renders inline in any browser cell without a tile-server install.

**`pyproj`** — Python bindings to the PROJ library, which is the C-level reference implementation of every coordinate transform you'll ever need. When you write `Transformer.from_crs(...)` you're calling decades of geodetic engineering. Use this for every reproject; never invent your own.


In [ ]:
# Install required packages
!pip install -q leafmap[common] pyproj geopandas shapely


## The approach (and why)

We're going to take the simplest possible space-GIS quantity — the sub-satellite point of the ISS at this moment — and prove that we can move it cleanly between two coordinate systems: WGS84 lat/lon (`EPSG:4326`, what GPS reports) and Web Mercator meters (`EPSG:3857`, what slippy maps draw in).

**Why this exact exercise?** It's the smallest possible workflow that exercises the core skill (CRS transformation) on real space-domain data (the ISS) using production-grade tools (`pyproj`). If this clicks, every harder transformation in the next 29 weeks is a variation on this theme.

**Why `always_xy=True`?** Half the GIS world passes `(lon, lat)` and half passes `(lat, lon)`. `pyproj` historically used the CRS's native axis order, which means WGS84 transformers took `(lat, lon)` while Web Mercator took `(x, y)`. The flag forces consistent `(lon, lat)` order for input AND output. Set it always.


In [ ]:
# Week 1: plot the ISS sub-satellite point on an interactive map.
import leafmap.foliumap as leafmap
from pyproj import Transformer

# Public ISS TLE (from CelesTrak — refresh with a real fetch in production)
lat, lon = 41.45, -73.79  # approximate ISS sub-satellite point

# Convert WGS84 lat/lon -> Web Mercator meters
transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
x, y = transformer.transform(lon, lat)
print(f"ISS sub-satellite point: ({lat:.4f}, {lon:.4f}) lat/lon")
print(f"In Web Mercator meters: ({x:.0f}, {y:.0f})")

m = leafmap.Map(center=[lat, lon], zoom=3)
m.add_marker((lat, lon), popup="ISS (sub-satellite point)")
m

# TODO: fetch the LIVE ISS TLE from CelesTrak and compute the sub-satellite
# point with skyfield instead of the hardcoded value above. See the primer.


## What just happened — and why it works

The map you see is rendered by **leafmap** using Folium/Leaflet under the hood. Internally, every tile fetched from the basemap is in Web Mercator — but you give leafmap WGS84 lat/lon, and it does the projection for you. That's why the marker shows up at the right spot even though the tiles are in different units.

The `transformer.transform(lon, lat)` call did the geodetic math: it converted a point on the WGS84 ellipsoid to its position in the Web Mercator plane (which is a Mercator projection assuming Earth is a perfect sphere — not the WGS84 ellipsoid). For a city-scale lookup that simplification is fine; for ranging a spacecraft we'd use a different CRS.


## Common gotchas

- **Mixed `(lat, lon)` vs `(lon, lat)` order.** Always set `always_xy=True` and always pass `(longitude, latitude)`. If your point lands in the Indian Ocean when you meant New York, this is the bug.
- **EPSG:4326 is geographic, not projected.** Distances measured in degrees are non-uniform: 1° of longitude at the equator is ~111 km, at 60° latitude it's ~55 km. Never compute Euclidean distance on lat/lon directly — project first, or use `pyproj.Geod` for geodesic distance.
- **Web Mercator falls apart near the poles.** At ±85° latitude it explodes. For Arctic / Antarctic work you want polar stereographic (EPSG:3413 / 3031).


## Doing this in QGIS (alternative path)

If you prefer desktop GIS, the QGIS path is **File → New Project → Layer → Add Delimited Text Layer**, then paste in the lat/lon and tell QGIS the CRS is EPSG:4326. To reproject visually: right-click the layer → Export → Save Features As, set the target CRS to EPSG:3857, save as GeoPackage. The point will land in the same geographic spot but the coordinate values in the attribute table will change to meters.

**Why use Python and not QGIS for this?** Reproducibility. A Python script with `pyproj` is one file, version-controllable, runnable in CI. A QGIS project is a binary `.qgz` with absolute file paths. Use QGIS for *exploration and final map composition*; use Python for *anything that needs to run twice*.


## Self-check

Verify your solution against these checks before considering the lab complete:

- [ ] Output type matches the expected format (GeoJSON / PNG / table / etc.).
- [ ] No exceptions raised on a clean run.
- [ ] Result is visually plausible — open the map cell, scan the values, sanity-check magnitudes.
- [ ] Quiz on the [week page](https://launchdetect.com/academy/week/1/) — try answering before checking the key.

---

Found a bug or want to contribute an improvement? Open an issue or PR at [github.com/launchdetect/academy-labs](https://github.com/launchdetect/academy-labs).
